# - 체인의 구조를 고도화하기, Runnable
### 필요 패키지 설치
- 아래 코드 셀을 실행하여 필요한 패키지를 설치합니다.

In [1]:
# %pip install -U langchain langchain-openai

In [2]:
%pip show langchain
%pip show langchain-openai

Name: langchain
Version: 1.3.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\workspaces\ai_agent\src\.venv\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.
Name: langchain-openai
Version: 1.5.2
Summary: An integration package connecting OpenAI and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/openai
Author: 
Author-email: 
License: MIT
Location: c:\workspaces\ai_agent\src\.venv\Lib\site-packages
Requires: certifi, langchain-core, openai, tiktoken
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv

load_dotenv()

True

## 1. RunnableParallel
> langchain_core.runnables 모듈의 RunnableParallel 클래스를 사용하여 두 가지 이상의 Pipeline을 병렬적으로 실행 가능

#### 개별 체인 정의
- 병렬로 실행할 체인을 정의합니다.

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI


prompt1 = PromptTemplate.from_template("{topic}를 주제로 농담 하나 해줘")
prompt2 = PromptTemplate.from_template("{topic}를 주제로 시 하나 작성해 줘")

model = ChatOpenAI(model="gpt-5-nano")

parser = StrOutputParser()

joke_chain = prompt1 | model | parser
poem_chain = prompt2 | model | parser

#### RunnableParallel 객체 생성 및 체인 할당
- `RunnableParallel` 클래스를 불러와 객체를 생성합니다

In [5]:
# ‘친구’라는 공통 주제에 대해서 각각 농담과 시를 작성하는 joke_chain과 poem_chain을 만들고, 이 두 체인을 동시에 실행시켜 봄
from langchain_core.runnables import RunnableParallel

map_chain = RunnableParallel(joke=joke_chain, poem=poem_chain)

#### 체인 실행  


In [6]:
# 두개의 chain을 동시에 실행
map_chain.invoke({'topic': '친구'})


{'joke': '좋은 친구를 주제로 한 짧은 농담 하나 드릴게요.\n\n우리 우정은 와이파이 같아서, 신호가 약해질 때는 서로를 더 가까이 안아주고, 연결될 때는 배꼽 빠지게 서로 웃겨요. \n\n더 원하시면 분위기(귀여운 말장난, 슬랩스틱, 짧은 한 줄 등) 알려주면 다른 버전도 바로 만들어 드릴게요.',
 'poem': '친구라는 길\n\n친구란 길 위의 등불, 매일 다른 그림자를 만들어도\n네가 걸을 때마다 내 발걸음은 더 가볍다.\n우리는 같은 하늘의 아래 서로 다른 창을 열고 닫는다.\n길 모퉁이에서 나눈 웃음은 비처럼 떨어져도 마르지 않는다.\n네가 속삭이던 비밀은 바람에 흩어져도 다시 모인다.\n먼지 쌓인 기억도 네 손길 한 번에 반짝인다.\n우리가 쓴 약속은 종이 위의 좌표 같아, 한참을 찾아도 길을 잃지 않는다.\n멀리 있어도 서로의 벽을 기댈 수 있게 한다.\n남몰래 눈물을 훔치던 밤에도 네가 곁에 있어 주는 건 축복이다.\n시간이 지나면 말의 색이 바래도, 함께한 침묵은 선명하다.\n너는 내게 바람의 방향을 가르쳐 준 사람,\n또 내가 길을 잃을 때 두 손으로 길을 다시 펴는 사람.\n그래서 고맙다고, 말 없이도 느낄 수 있다.\n친구야, 이 거리는 너를 닮아 더 따뜻해진다.\n우리가 남긴 발자국은 한때의 걱정을 덜어 주는 작은 불씨가 된다.\n그리고 오늘도 밤하늘 아래 서로의 이름을 속삭인다.'}

#### (실습) RunnableParallel 응용

요구사항 1: 영화 질문 답변 및 Prompt 생성
1. 사용자는 영화와 관련된 질문을 입력합니다.
2. Chain 1 을 통해 사용자 질문에 대한 답변(answer)을 받습니다.
3. RunnableParallel을 이용하여 두 작업을 동시에 수행합니다:
    * Chain 2-1 : 추천 프롬프트(prompt_recommend):
        * 사용자의 질문과 AI의 답변을 바탕으로 사용자가 새롭게 질문할 만한 추천 프롬프트 3개를 생성하시오.
    * Chain 2-2 : 유사 질문(sim_question):
        * 사용자의 질문과 유사하지만 사용된 단어가 다른 새로운 질문 3개를 생성하시오.

요구사항 2: 리스트 형식으로 반환
1. 2-1, 2-2 Chain은 모두 CommaSeparatedListOutputParser를 사용해 리스트 형태로 반환합니다.
2. 최종 출력은 {"prompt_recommend": [...], "sim_question": [...]} 형태여야 합니다. (dictionary 포맷)


In [26]:
question = "인셉션 감독이 누구인가요?"

In [27]:
template1 = """
당신은 영화를 추천해주는 AI 챗봇입니다. User의 질문에 대해 답변하시오.
question : {question}
"""

prompt = PromptTemplate(
    template=template1
)

model = ChatOpenAI(model_name="gpt-5-nano")

chain1 = prompt | model | StrOutputParser()

answer = chain1.invoke({"question" : question})

print(answer)

인셉션의 감독은 크리스토퍼 놀란(Christopher Nolan)입니다. 이 영화는 2010년에 개봉했으며 놀란이 감독과 각본을 맡았습니다. 더 알고 싶거나 비슷한 분위기의 영화 추천이 필요하면 말씀해 주세요.


In [28]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()
# 출력 형식 지침
format_instructions = output_parser.get_format_instructions()

# Chain 2-1 : 추천 프롬프트(prompt_recommend):
recommend_template = """
당신은 영화를 추천해 주는 AI 챗봇입니다. User의 질문과 AI의 답변 다음으로 User가 질문할 만한 추천 프롬프트 3개를 말해주세요.
User_Message : {question}
AI_Message : {answer}

FORMAT :
{format}
"""

recommend_prompt = PromptTemplate(
    template=recommend_template,
    partial_variables={
        "format": format_instructions
    },
)


In [29]:
# Chain 2-2 : 유사 질문(sim_question): 증강 질문 생성

augmented_template = """
당신은 사용자의 질문을 새롭게 생성하는 인공지능 비서입니다. User의 질문과 유사하지만 사용한 단어는 다른 질문 3개를 말해주세요.
User_Message : {question}

FORMAT :
{format}
"""

augmented_prompt = PromptTemplate(
    template=augmented_template,
    partial_variables={
        "format": format_instructions
    },
)

model = ChatOpenAI(model="gpt-5-nano")


In [ ]:
# 두개의 chain 연결 실행

recommend_response = recommend_prompt | model | output_parser  # 추천 질문
augmented_question = augmented_prompt | model | output_parser  # 증강 질문

In [31]:
# 두개의 chain RunnableParallel로 연결
from langchain_core.runnables import RunnableParallel

map_chain = RunnableParallel(
    recommend_response= recommend_response, # 추천 질문
    augmented_question= augmented_question  # 증강 질문
)

result = map_chain.invoke({'question': question, 'answer': answer})

print(result)

{'recommend_response': ['인셉션과 비슷한 분위기의 영화 추천해줘', '크리스토퍼 놀란의 다른 작품 중에서 비슷한 연출 스타일의 영화 추천해줘', '시간과 꿈', '현실의 경계를 다루는 주제의 영화를 추천해줘'], 'augmented_question': ['인셉션의 연출가는 누구인가요?', '크리스토퍼 놀란은 이 영화의 감독인가요?', '인셉션의 감독 이름이 무엇인가요?']}


---

## 2. RunnableLambda

> RunnableLambda를 사용하여 사용자 정의 함수를 Chain에 맵핑할 수 있다.

#### 일반 함수와 lambda 함수

In [32]:
# 일반 함수
def double(x):
    return x * 2

# lambda 함수 예제
numbers = [1, 2, 3, 4]
double_numbers = list(map(lambda x: x * 2, numbers))
print(double_numbers) # [2, 4, 6, 8]

[2, 4, 6, 8]


### RunnableLambda 1
#### 함수 정의
- RunnableLambda에서 파이썬 기본 내장 함수 len을 바로 사용할 수도 있지만, 예시를 위함이니 간단하게 정의합니다.

In [33]:
def length_function(word):
    return len(word)

#### 기초 체인 정의
- 기본 프롬프트, 모델, 출력 파서를 정의합니다.

In [35]:
prompt = PromptTemplate.from_template(
    "{a} + {b}는 무엇인가요?"
)
model = ChatOpenAI(model="gpt-5-nano")
output_parser = StrOutputParser()

#### RunnableLambda를 통한 함수 연결 및 실행


In [ ]:
from langchain_core.runnables import RunnableLambda

# Chain 구성
chain = (
    {
        'a': RunnableLambda(lambda x: length_function(x['word1'])),
        'b': RunnableLambda(lambda x: length_function(x['word2']))
    } 
    | prompt 
    | model 
    | output_parser
)

chain.invoke({'word1': '안녕하세요.', 'word2': '반가워요.'})

'6 더하기 5는 11입니다.'

### RunnableLambda를 활용한 라우팅(분기)
#### classifier_chain 체인 정의
- 사용자의 질문에 대한 주제를 파악해서 한 단어로 답변하는 체인

In [38]:
template = """
주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요.
이외의 답변은 허용하지 않습니다.
<question>
{question}
</question>

<answer example>
수학
</answer example>
"""

classifier_prompt = PromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-5-nano")
output_parser = StrOutputParser()

classifier_chain = classifier_prompt | model | output_parser

result = classifier_chain.invoke({"question": "2+2 는 무엇인가요?"})

result

'수학'

#### 개별 체인 생성
- 각 주제에 특화된 math_chain, science_chain, general_chain을 정의합니다.

In [16]:
# 수학 체인
math_prompt = PromptTemplate.from_template(
        """
          당신은 수학 전문가입니다.
          항상 다음과 같이 답변을 시작합니다. "피타고라스께서 말씀하시기를…"


          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )

# 과학 체인
science_prompt = PromptTemplate.from_template(
        """
          당신은 과학 전문가입니다.\
          항상 다음과 같이 답변을 시작합니다. "뉴턴께서 말씀하시기를…"

          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )

# 일반 체인
general_prompt = PromptTemplate.from_template(
        """
          당신은 일반 상식 전문가입니다.\
          항상 다음과 같이 답변을 시작합니다. "부모님께서 말씀하시기를 …"

          다음 질문에 답변하시오:

          질문: {question}
          답변:
        """
    )


math_chain = math_prompt | model | output_parser
science_chain = science_prompt | model | output_parser
general_chain = general_prompt | model | output_parser

#### Route 함수 정의
- classifier_chain의 결과(주제)를 입력받아 조건에 따라 분기 처리하는 route 함수를 정의합니다.
- 단, 이때 이 route 함수가 언제 호출될 것이며, 이때 전달되는 값이 어떤 형식인지를 생각하여 정의해야 합니다.

In [39]:
# 함수 정의
def route(info):
    if '수학' in info['topic']:  # 입력으로 딕셔너리로 전달
        return math_chain
    elif '과학' in info['topic']:
        return science_chain
    else:
        return general_chain


#### full chain 연결
- 앞에서 정의한 함수와 각 체인들을 모두 결합하여 full_chain을 정의합니다.

In [ ]:
# 함수를 딕셔너리 형태로 chain과 연결
full_chain = (
    {'topic': classifier_chain}
    | RunnableLambda(route)   # 객체를 chain으로 연결 하면서 route 함수의 입력으로 전달
    | StrOutputParser()
)

full_chain.invoke({'question': '12+12 는 무엇인가요?'})   
# KeyError 발생 => 
# full_chain에서 invoke 메서드를 호출할 때, 사용자의 질문을 “question”에 담아 전달                             
# 그때, classifier_chain은 정상적으로 사용자의 질문을 받을 수 있지만, route 함수를 통해 실행될 각 유형별 체인은 사용자의 질문을 넘겨 받을 수 없음

#### KeyError 해결 후 최종 코드

In [41]:
# route 함수에 전달될 입력 딕셔너리를 정의하는 것
full_chain = (
    {
        'topic': classifier_chain,
        'question': lambda x: x['question']
    }
    | RunnableLambda(route)   # 객체를 chain으로 연결 하면서 route 함수의 입력으로 전달
    | StrOutputParser()
)

full_chain.invoke({'question': '12+12 는 무엇인가요?'})   


'피타고라스께서 말씀하시기를… 12+12는 24입니다.'

#### 실습: RunnableLambda-router

사용자가 질문을 던지면, 챗봇은 이를 **유튜브, 동물병원, 보험, 기타**로 분류한 후 적절한 체인으로 라우팅하여 응답합니다.  RunnableLambda, PromptTemplate, 및 체인 연결을 활용하여 동작하는 AI 시스템을 구현하시오.

**요구사항**
1. 입력 데이터:
    * 사용자는 질문(question)을 입력합니다.
2. 라우터:
    * 주어진 질문을 유튜브, 동물병원, 보험, 기타 중 하나로 분류해야 합니다.
    * router_chain에서 질문을 분석하고 결과를 topic 키의 값으로 반환합니다.
3. 라우팅 로직:
    * **RunnableLambda**를 사용하여 topic 값에 따라 적절한 체인을 선택하세요:
        * 유튜브 → youtube_chain
        * 동물병원 → hospital_chain
        * 보험 → insurance_chain
        * 기타 → general_chain
4. 최종 결과 출력:
    * 선택된 체인의 결과를 문자열로 반환하며, 최종 응답을 출력합니다.


In [17]:
# router_chain의 invoke한 값에 따라 chain 호출


In [18]:
# 유튜브 체인


# 동물병원 체인


# 보험 체인


# 일반 체인


In [19]:
# 전체 체인 정의


---

## 3. RunnablePassthrough
- 원활한 실습 진행을 위한 classifier_chain 재정의

- 아래 코드를 먼저 실행하고 다음 과정으로 넘어갑니다.


In [42]:

template = """
주어진 사용자 질문을 `수학`, `과학`, 또는 `기타` 중 하나로 분류하세요.
이외의 답변은 허용하지 않습니다.
<question>
{question}
</question>

<answer example>
수학
</answer example>
"""

classifier_prompt = PromptTemplate.from_template(template)
model = ChatOpenAI(model="gpt-5-nano")
output_parser = StrOutputParser()

classifier_chain = classifier_prompt | model | output_parser

result = classifier_chain.invoke({"question": "2+2 는 무엇인가요?"})

result

'수학'

#### 데이터 흐름 테스트
- 마찬가지로, 아래 코드를 실행하여 전체 데이터의 흐름을 파악합니다.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# route 함수에 전달될 입력 딕셔너리를 정의하는 것
full_chain = (
    {
        'topic': classifier_chain,
        'question': lambda x: x['question']
    }
    | RunnableParallel(        # 객체를 병렬 chain으로 연결 
        passed = RunnablePassthrough(), # by pass 역활
        modified = RunnableLambda(route) # route 함수 호출
    )  
)

full_chain.invoke({'question': '12+12 는 무엇인가요?'})   


{'passed': {'topic': '수학', 'question': '12+12 는 무엇인가요?'},
 'modified': '피타고라스께서 말씀하시기를… 12와 12를 더하면 24가 됩니다.'}

#### RunnablePassthrough.assign()

In [ ]:
full_chain = (
    RunnablePassthrough.assign(topic=classifier_chain)  # 매개변수로 전달
    | RunnableLambda(route)
)

# KeyError 발생 없이 실행 
# assign 메서드를 사용하면 기존에 전달한 딕셔너리에 새로운 키-값 쌍을 추가해 딕셔너리를 확장할 수 있음
# 이전 체인에서 얻은 데이터를 다음 체인으로 전달할 때, 원본 입력 데이터가 소실되는 문제를 해결
full_chain.invoke({'question': '12+12 는 무엇인가요?'})   


'피타고라스께서 말씀하시기를… 12와 12를 더하면 24가 됩니다.'